<a href="https://colab.research.google.com/github/gabrielcarcedo/Multimodal-GNN-for-Chagas-Classification/blob/main/GAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Multimodal Graph Learning for Chagas Disease Classification**
## *Infection Stage Classification via Graph Attention Networks (GAT)*

**Authors:** Gabriel Carcedo-Rodríguez, Erik Molino-Minero-Re, Jorge Perez-Gonzalez, Nidiyare Hevia-Montiel.

**Description:**
This notebook implements the core classification and interpretability phase of our research. It utilizes Graph Attention Networks (GAT) to process fully connected multimodal graphs—incorporating both original murine data and synthetic subjects generated by the VGAE—to classify the stages of *Trypanosoma cruzi* infection. The GAT architecture dynamically calculates attention coefficients to prioritize the most relevant clinical biomarkers. Furthermore, this notebook includes the implementation of `GNNExplainer` for *post-hoc* interpretability, allowing us to evaluate and validate the pathophysiological consistency of the network's predictions.

# **Install dependencies**

In [ ]:
!pip install -q torch-geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install -q pandas numpy matplotlib seaborn scikit-learn

# **Import Packages and Libraries**

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, GCNConv
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import os
import csv
from itertools import product
from google.colab import files
import torch.optim.lr_scheduler as lr_scheduler
from torch_geometric.explain import Explainer, GNNExplainer, ExplainerConfig, ModelConfig
import random

def set_seed(seed=100):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(100)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Usando dispositivo: cpu


# **Preprocess Data**

In [ ]:
def load_and_preprocess_data(csv_path):
    db = pd.read_csv(csv_path)
    db.set_index('ID paciente', inplace=True)
    db.dropna(inplace=True)

    cols_to_drop = ['lgGT control', 'lgG1 control', 'lgG2a control']
    db = db.drop(columns=[c for c in cols_to_drop if c in db.columns])

    cols_echo = ['HR SHORTH AXIS','LVd','LVs','FS','EF']
    cols_elisa = ['lgGT','lgG1','lgG2a']
    cols_ecg = ['HR', 'HRV', 'CV', 'RR Interval', 'PQ Interval', 'PR Interval', 'QRS',
                'QT Interval', 'ST Interval', 'QTc Interval', 'QTc', 'QTc dispersion',
                'SR mean', 'R Amplitude mean']
    cols_doppler = ['AbAO HR avg', 'AbAO HR SD', 'AbAO RR internal Avg', 'AbAO RR interval SD',
                    'AbAO Peak velocity Avg', 'AbAO Peak velocity SD', 'AbAO Minimum Flow Velocity Avg',
                    'AbAO Minimum Velocity SD', 'AbAO Mean Flow velocity Avg', 'AbAO Mean Flow velocity SD',
                    'AbAO Pulsability Index Avg', 'AbAO Pulsability Index SD', 'AbAO Resistivity Index Avg',
                    'AbAO Resistivity Index SD', 'AO HR Avg', 'AO HR SD', 'AO RR Interval Avg',
                    'AO RR Interval SD', 'AO Pre-ejection time Avg', 'AO Pre-ejection time SD',
                    'AO Peak velocity Avg', 'AO Peak velocity SD', 'AO Stroke Distance Avg',
                    'AO Stroke Distance SD', 'AO Ejection time Avg', 'AO Ejection Time SD',
                    'AO Rise Time Avg', 'AO Rise Time SD', 'AO Mean velocity Avg', 'AO Mean velocity SD',
                    'AO Mean Acceleration Time Avg', 'AO Mean Acceleration Time SD', 'AO Peak Acceleration Avg',
                    'AO Peak Acceleration SD', 'MV HR Avg', 'MV HR SD', 'MV RR Interval Avg',
                    'MV RR Interval SD', 'MV E Peak velocity Avg', 'MV E Peak velocity SD',
                    'MV E Acceleration time Avg', 'MV E Acceleration time SD', 'MV E Peak to Peak time SD',
                    'MV E Deceleration time SD', 'MV E Deceleration Rate SD']

    def standardize_block(df, columns):
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(df[columns])
        return pd.DataFrame(data_scaled, columns=columns, index=df.index)

    db_echo = standardize_block(db, cols_echo)
    db_elisa = standardize_block(db, cols_elisa)
    db_ecg = standardize_block(db, cols_ecg)
    db_doppler = standardize_block(db, cols_doppler)

    db_all = pd.concat([db_echo, db_elisa, db_ecg, db_doppler], axis=1)
    db_all['Label'] = db['Label'].values
    db_all.loc[db_all['Label'] == 2, 'Label'] = 1

    return db_all

# **Graph Attention Network (GAT)** Model

In [ ]:
class GAT_Model_Attention(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=1, dropout=0.5):
        super(GAT_Model_Attention, self).__init__()
        self.dropout_ratio = dropout
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, concat=True, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False, dropout=dropout)
        self.fc = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch, return_attention=False):
        x = F.dropout(x, p=self.dropout_ratio, training=self.training)
        if return_attention:
            x, (att_edge_index, att_weights) = self.conv1(x, edge_index, return_attention_weights=True)
        else:
            x = self.conv1(x, edge_index)
        x = F.elu(x)

        x = F.dropout(x, p=self.dropout_ratio, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.elu(x)

        out = global_mean_pool(x, batch)
        out = self.fc(out)

        if return_attention:
            return out, att_edge_index, att_weights
        else:
            return out

## Graphs Cronstruction

In [ ]:
def create_gat_graphs(df):
    graphs = []
    feature_cols = df.columns.drop('Label')
    num_nodes = len(feature_cols)

    nodes = range(num_nodes)
    edge_index = torch.tensor(list(product(nodes, nodes)), dtype=torch.long).t().contiguous()
    node_identity = torch.eye(num_nodes)

    for i, row in df.iterrows():
        label = row['Label']
        subject_id = row.name

        values = torch.tensor(row[feature_cols].values, dtype=torch.float).view(-1, 1)
        x = torch.cat([values, node_identity], dim=1)

        graph = Data(
            x=x,
            edge_index=edge_index,
            y=torch.tensor([label], dtype=torch.long)
        )
        graph.subject_id = [str(subject_id)]

        graphs.append(graph)

    return graphs, num_nodes

## Explain Patient Prediction

In [ ]:
def explain_patient_prediction(model, dataset_loader, subject_id, feature_names, device, subject_type):
    target_graph = None
    for graph in dataset_loader.dataset:
        if graph.subject_id == subject_id:
            target_graph = graph
            break

    if target_graph is None:
        print("Sujeto no encontrado.")
        return

    data = target_graph.to(device)
    batch = torch.zeros(data.x.size(0), dtype=torch.long).to(device)

    explainer = Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=200),
        explanation_type='model',
        node_mask_type='object',
        edge_mask_type='object',
        model_config=dict(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )

    model.eval()
    explanation = explainer(x=data.x, edge_index=data.edge_index, batch=batch)

    node_importance = explanation.node_mask.cpu().detach().numpy().flatten()

    if node_importance.max() > 0:
        node_importance = node_importance / node_importance.max()

    sorted_idx = np.argsort(node_importance)[::-1]
    sorted_features = np.array(feature_names)[sorted_idx]
    sorted_values = node_importance[sorted_idx]

    plt.figure(figsize=(10, 6))

    top_k = 15
    ax = sns.barplot(x=sorted_values[:top_k], y=sorted_features[:top_k], palette="rocket_r")
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)

    with torch.no_grad():
        out = model(data.x, data.edge_index, batch)
        pred_class = out.argmax(dim=1).item()
        prob = torch.softmax(out, dim=1)[0, pred_class].item()

    clase_str = f"{subject_type}" if pred_class == 1 else "Control"

    plt.title(f"GNNExplainer: Top Features for {subject_id}\nPred: {clase_str} ({prob:.1%})", fontsize=20, fontweight='bold')
    plt.xlabel("GNNExplainer Score", fontsize=16, fontweight='bold')
    plt.ylabel("Feature", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return sorted_features, sorted_values

## **GAT** Train Function

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_probs = []
    all_labels = []

    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        probs = F.softmax(out, dim=1)
        all_probs.append(probs.detach().cpu().numpy())
        all_labels.append(data.y.detach().cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    preds = np.argmax(all_probs, axis=1)
    acc = (preds == all_labels).mean()

    try:
        if len(np.unique(all_labels)) > 1:
             auc = roc_auc_score(label_binarize(all_labels, classes=[0,1]), all_probs[:, 1])
        else:
            auc = 0.5
    except:
        auc = 0.5

    return total_loss / len(loader), acc, auc

def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    all_ids = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            probs = F.softmax(out, dim=1)
            pred = out.argmax(dim=1)

            all_preds.append(pred.cpu().numpy())
            all_labels.append(data.y.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

            if isinstance(data.subject_id, list):
                all_ids.extend(data.subject_id)
            else:
                all_ids.extend(data.subject_id)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)

    acc = (all_preds == all_labels).mean()

    try:
        if len(np.unique(all_labels)) > 1:
            auc = roc_auc_score(label_binarize(all_labels, classes=[0,1]), all_probs[:, 1])
        else:
            auc = 0.5
    except:
        auc = 0.5

    return acc, auc, all_labels, all_preds, all_probs, all_ids

## Visualizar resultados de entrenamiento

In [ ]:
def plot_training_results(history, cm, labels_map, folder_name, fold_k):
    plt.figure(figsize=(18, 4))

    # Loss
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train Loss', color='orange', alpha=0.5)
    plt.scatter(np.arange(len(history['train_loss'])), history['train_loss'], color='orange', linewidths=1)
    plt.title(f'Fold {fold_k}: Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(history['train_acc'], label='Train Acc', color='blue', alpha=0.7)
    plt.plot(history['val_acc'], label='Val Acc', color='green', lw=3)
    plt.title(f'Fold {fold_k}: Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # AUROC
    plt.subplot(1, 3, 3)
    plt.plot(history['train_auc'], label='Train AUC', color='blue', alpha=0.7)
    plt.plot(history['val_auc'], label='Val AUC', color='green', lw=3)
    plt.title(f'Fold {fold_k}: AUROC')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Confusion Matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels_map, yticklabels=labels_map)
    plt.title(f'Confusion Matrix - Fold {fold_k}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

## Load Synthetic Data

In [ ]:
def get_synthetic_data(filename, mode, feature_cols):
    if not os.path.exists(filename):
        print(f"{filename} don't find")
        return None

    df_syn = pd.read_csv(filename)

    if 'Sujeto' in df_syn.columns:
        df_syn.set_index('Sujeto', inplace=True)

    missing_cols = [c for c in feature_cols if c not in df_syn.columns]
    if missing_cols:
        print(f"Error: Missing columns {missing_cols}")
        return None

    df_syn = df_syn[feature_cols].copy()

    if mode == 'CONTROL_VS_ACUTE':
        df_subset = df_syn.iloc[0:30].copy()
        print(f"   -> Load 30 instances (Control+Acute)")

    elif mode == 'CONTROL_VS_CHRONIC':
        df_subset = df_syn.iloc[30:60].copy()
        df_subset['Label'] = df_subset['Label'].replace({2: 1})
        print(f"   -> Load 30 instances  (Control+Chronic)")

    elif mode == 'CONTROL_VS_INFECTION':
        df_subset = df_syn.copy()
        df_subset['Label'] = df_subset['Label'].replace({2: 1})
        print(f"   -> Load 60 instances  (Todos)")

    else:
        return None

    return df_subset

## **Training** *Example*

In [ ]:
EXPERIMENT_MODE = 'CONTROL_VS_ACUTE' # 'CONTROL_VS_ACUTE', 'CONTROL_VS_CHRONIC', 'CONTROL_VS_INFECTION'
MODALIDAD = 'MULTIMODAL_VOTING_VGAE_FS'
subject_type = 'Acute'
USE_SYNTHETIC = True

# Hyperparameters
K_FOLDS = 5
EPOCHS = 2000
PATIENCE = 200
BATCH_SIZE = 4
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01
HIDDEN_CHANNELS = 64
HEADS = 2
DROPOUT = 0

# Load database
if 'db_all' not in globals():
    filename = 'bd_final_multimodal_120821.csv'
    if os.path.exists(filename):
        db_all = load_and_preprocess_data(filename)
    else:
        raise FileNotFoundError("Sube el archivo CSV primero.")

# Features employed
features_subset = ['LVd', 'AO Pre-ejection time Avg',
       'AO Ejection Time SD', 'AO Peak Acceleration Avg', 'MV HR Avg',
       'HR', 'HRV', 'CV', 'QT Interval', 'ST Interval', 'QTc', 'SR mean',
       'lgGT', 'lgG1', 'lgG2a', 'Label']
df_base = db_all[features_subset].copy()

# Instances Selection
if EXPERIMENT_MODE == 'CONTROL_VS_ACUTE':
    print(f"--- MODO: {EXPERIMENT_MODE} (Primeros 36 registros) ---")
    df_exp = df_base.iloc[:36].copy()
    class_names = ['Control', 'Acute']

elif EXPERIMENT_MODE == 'CONTROL_VS_CHRONIC':
    print(f"--- MODO: {EXPERIMENT_MODE} (Últimos 36 registros) ---")
    df_exp = df_base.iloc[36:].copy()
    df_exp['Label'] = df_exp['Label'].replace({2: 1})
    class_names = ['Control', 'Chronic']

elif EXPERIMENT_MODE == 'CONTROL_VS_INFECTION':
    print(f"--- MODO: {EXPERIMENT_MODE} (Todos los registros) ---")
    df_exp = df_base.iloc[:].copy()
    df_exp['Label'] = df_exp['Label'].replace({2: 1})
    class_names = ['Control', 'Infected']

print(f"Muestras totales: {len(df_exp)}")
print(f"Distribución: {df_exp['Label'].value_counts().to_dict()}\n")

# Load Synthetic Data
df_synthetic = None
if USE_SYNTHETIC:
    df_synthetic = get_synthetic_data('datos_sintéticos_features_selection.csv', EXPERIMENT_MODE, features_subset)

# Result Folder
folder_name = f"{MODALIDAD}_{EXPERIMENT_MODE}_results"
os.makedirs(folder_name, exist_ok=True)

# Principal Loop
results = []

for k in range(1, K_FOLDS + 1):
    print(f'\n{"="*60}')
    print(f'FOLD {k}/{K_FOLDS} - Comparando clases {class_names}')
    print(f'{"="*60}')


    # Split Test (6 instances)
    train_val_df, test_df = train_test_split(df_exp, test_size=6, stratify=df_exp['Label'], random_state=k)

    # Split validation (6 instances)
    train_df_real, val_df = train_test_split(train_val_df, test_size=6, stratify=train_val_df['Label'], random_state=k)

    # DATA AUGMENTATION: Real + Synthetic
    if df_synthetic is not None:
        train_df = pd.concat([train_df_real, df_synthetic], axis=0)
        print(f"   -> Train Size: {len(train_df)} ({len(train_df_real)} reales + {len(df_synthetic)} sintéticos)")
    else:
        train_df = train_df_real
        print(f"   -> Train Size: {len(train_df)} (Solo reales)")

    print(f"   -> Val Size: {len(val_df)} (Reales) | Test Size: {len(test_df)} (Reales)")

    # Graph Generation
    train_graphs, num_nodes = create_gat_graphs(train_df)
    val_graphs, _ = create_gat_graphs(val_df)
    test_graphs, _ = create_gat_graphs(test_df)

    # DataLoaders
    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

    # GAT Model
    model = GAT_Model_Attention(in_channels=1 + num_nodes,
                      hidden_channels=HIDDEN_CHANNELS,
                      out_channels=2,
                      heads=HEADS,
                      dropout=DROPOUT).to(device)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    # Scheduler
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=25)

    # Results audit
    history = {'train_loss': [], 'train_acc': [], 'train_auc': [], 'val_acc': [], 'val_auc': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_model_wts = None
    patience_counter = 0

    # Training
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, train_auc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_acc, val_auc, _, _, _, _ = evaluate(model, val_loader, device)

        # Val Loss
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for d in val_loader:
                d = d.to(device)
                out = model(d.x, d.edge_index, d.batch)
                val_loss += criterion(out, d.y).item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_auc'].append(train_auc)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)
        history['val_loss'].append(val_loss)

        if epoch == 1 or epoch % 100 == 0:
            curr_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch}/{EPOCHS}, Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, LR: {curr_lr:.6f}")

        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    # Evaluation
    if best_model_wts:
        model.load_state_dict(best_model_wts)
        torch.save(model.state_dict(), f'{folder_name}/best_model_fold{k}.pth')

    test_acc, test_auc, y_true, y_pred, probs, ids = evaluate(model, test_loader, device)

    # Graphics
    cm = confusion_matrix(y_true, y_pred)
    plot_training_results(history, cm, class_names, folder_name, k)

    print("\nReporte de Clasificación:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    print("\nPredicciones vs Reales:")
    y_true_names = [class_names[i] for i in y_true]
    y_pred_names = [class_names[i] for i in y_pred]
    for sujeto_id, real_label, pred_label in zip(ids, y_true_names, y_pred_names):
        status = "✅" if real_label == pred_label else "❌"
        print(f"Sujeto: {sujeto_id} | Real: {real_label:8s} | Pred: {pred_label:8s} {status}")

    # GNNExpleiner
    if len(test_loader.dataset) > 0:
        example_id = test_loader.dataset[id_graph].subject_id
        feat_names_clean = [f for f in features_subset if f != 'Label']
        top_feats, top_vals = explain_patient_prediction(model, test_loader, example_id, feat_names_clean, device, subject_type)

    # Results
    best_idx = np.argmin(history['val_loss'])

    best_train_acc_fold = np.max(history['train_acc'])
    best_train_auc_fold = np.max(history['train_auc'])
    best_val_acc_fold = np.max(history['val_acc'])
    best_val_auc_fold = np.max(history['val_auc'])

    results.append({
        'Fold': k,
        'Train Acc': best_train_acc_fold,
        'Train AUC': best_train_auc_fold,
        'Val Acc': best_val_acc_fold,
        'Val AUC': best_val_auc_fold,
        'Test Acc': test_acc,
        'Test AUC': test_auc
    })

    print(f"\nFold {k} completado | Train Acc: {best_train_acc_fold:.4f} | Train AUC: {best_train_auc_fold:.4f} | Val Acc: {best_val_acc_fold:.4f} | Val AUC: {best_val_auc_fold:.4f} | Test Acc: {test_acc:.4f} | Test AUC: {test_auc:.4f}")
    torch.cuda.empty_cache()

# Summary after 5 folds
print(f'\n{"="*60}')
print("RESUMEN PROMEDIO DE LOS 5 FOLDS")
print(f'{"="*60}')

def get_stats(key):
    values = [r[key] for r in results]
    return np.mean(values), np.std(values)

train_acc_mean, train_acc_std = get_stats('Train Acc')
train_auc_mean, train_auc_std = get_stats('Train AUC')
val_acc_mean, val_acc_std = get_stats('Val Acc')
val_auc_mean, val_auc_std = get_stats('Val AUC')
test_acc_mean, test_acc_std = get_stats('Test Acc')
test_auc_mean, test_auc_std = get_stats('Test AUC')

print(f"{'Metric':<15} | {'Mean':<10} | {'Std Dev':<10}")
print("-" * 40)
print(f"{'Train Acc':<15} | {train_acc_mean:.4f}     | {train_acc_std:.4f}")
print(f"{'Train AUROC':<15} | {train_auc_mean:.4f}     | {train_auc_std:.4f}")
print("-" * 40)
print(f"{'Val Acc':<15} | {val_acc_mean:.4f}     | {val_acc_std:.4f}")
print(f"{'Val AUROC':<15} | {val_auc_mean:.4f}     | {val_auc_std:.4f}")
print("-" * 40)
print(f"{'Test Acc':<15} | {test_acc_mean:.4f}     | {test_acc_std:.4f}")
print(f"{'Test AUROC':<15} | {test_auc_mean:.4f}     | {test_auc_std:.4f}")
print("-" * 40)

# Save results
df_res = pd.DataFrame(results)
df_res.loc['Mean'] = df_res.mean()
df_res.loc['Std'] = df_res.std()
cols_order = ['Fold', 'Train Acc', 'Train AUC', 'Val Acc', 'Val AUC', 'Test Acc', 'Test AUC']
df_res = df_res[cols_order]

detailed_csv_path = f"{folder_name}/{MODALIDAD}_{EXPERIMENT_MODE}_stats.csv"
df_res.to_csv(detailed_csv_path)
print(f"\nReporte detallado guardado en: {detailed_csv_path}")

master_log_path = "bitacora_experimentos_global.csv"
row_data = {
    'Timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'Modalidad': MODALIDAD,
    'Experiment': EXPERIMENT_MODE,
    'Model': 'GAT (Connected)',
    'Train_Acc_Mean': train_acc_mean, 'Train_Acc_Std': train_acc_std,
    'Val_Acc_Mean': val_acc_mean,   'Val_Acc_Std': val_acc_std,
    'Test_Acc_Mean': test_acc_mean,  'Test_Acc_Std': test_acc_std,
    'Test_AUC_Mean': test_auc_mean,  'Test_AUC_Std': test_auc_std
}
df_log_new = pd.DataFrame([row_data])

if os.path.exists(master_log_path):
    df_log_new.to_csv(master_log_path, mode='a', header=False, index=False)
else:
    df_log_new.to_csv(master_log_path, mode='w', header=True, index=False)

print(f"Bitácora global actualizada en: {master_log_path}")
print("-" * 60)

# Download results
!zip -r {folder_name}.zip {folder_name}
files.download(f'{folder_name}.zip')